In [21]:
import numpy as np
import cv2 as cv


In [22]:

def video_to_1darray(path):
    frames = []
    video = cv.VideoCapture(path)
    while True:
        read, frame= video.read()
        if not read:
            break
        gray_frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
        frames.append(gray_frame)
    frames = np.array(frames)
    A = frames.reshape(frames.shape[0], -1).T
    return A


A = video_to_1darray("322.mp4")

In [23]:
A.shape

(307200, 1501)

Flattening frames: 3D -> 2D

In [24]:
A = frames.reshape(frames.shape[0], -1).T
A.shape

(307200, 1501)

IALM RPCA core: solve `A = L + S`

In [25]:
def solve_rpca(
    A,
    lam=None,
    mu=None,
    rho=1.5,
    max_iter=500,
    tol=1e-7,
    return_history=False,
    verbose=False,
):
    """Solve Robust PCA using Inexact Augmented Lagrange Multiplier (IALM).

    Parameters
    ----------
    A : ndarray, shape (m, n)
        Input data matrix (pixels x frames).
    lam : float, optional
        Sparse penalty. Default is 1 / sqrt(max(m, n)).
    mu : float, optional
        Initial ALM penalty parameter.
    rho : float
        Multiplicative update factor for mu.
    max_iter : int
        Maximum number of iterations.
    tol : float
        Relative Frobenius residual tolerance.
    return_history : bool
        If True, also return list of residuals.
    verbose : bool
        If True, print optimization diagnostics.

    Returns
    -------
    L : ndarray
        Low-rank component (background).
    S : ndarray
        Sparse component (foreground).
    history : list[float], optional
        Relative residual history per iteration.
    """
    A = np.asarray(A, dtype=np.float64)
    if A.ndim != 2:
        raise ValueError("A must be a 2D matrix")

    m, n = A.shape
    if m == 0 or n == 0:
        raise ValueError("A must be non-empty")

    if lam is None:
        lam = 1.0 / np.sqrt(max(m, n))

    norm_two = np.linalg.norm(A, 2)
    if norm_two == 0:
        L = np.zeros_like(A)
        S = np.zeros_like(A)
        if return_history:
            return L, S, [0.0]
        return L, S

    norm_inf = np.max(np.abs(A)) / lam
    dual_norm = max(norm_two, norm_inf)
    Y = A / dual_norm

    if mu is None:
        mu = 1.25 / norm_two
    mu_bar = mu * 1e7

    d_norm = np.linalg.norm(A, "fro")
    L = np.zeros_like(A)
    S = np.zeros_like(A)
    history = []

    for k in range(1, max_iter + 1):
        # Low-rank update: Singular Value Thresholding
        temp = A - S + (1.0 / mu) * Y
        U, s, Vt = np.linalg.svd(temp, full_matrices=False)
        s_thresh = np.maximum(s - (1.0 / mu), 0.0)
        rank = int(np.sum(s_thresh > 0.0))

        if rank > 0:
            L = (U[:, :rank] * s_thresh[:rank]) @ Vt[:rank, :]
        else:
            L = np.zeros_like(A)

        # Sparse update: elementwise soft-thresholding
        temp = A - L + (1.0 / mu) * Y
        S = np.sign(temp) * np.maximum(np.abs(temp) - (lam / mu), 0.0)

        # Residual and convergence
        residual = A - L - S
        err = np.linalg.norm(residual, "fro") / d_norm
        history.append(err)

        if verbose and (k == 1 or k % 10 == 0 or err < tol):
            sparsity = np.mean(S != 0)
            print(f"iter={k:4d} err={err:.3e} rank(L)={rank} sparsity(S)={sparsity:.4f}")

        if err < tol:
            break

        # Dual and penalty updates
        Y = Y + mu * residual
        mu = min(mu * rho, mu_bar)

    if return_history:
        return L, S, history
    return L, S


In [26]:
# Run RPCA on a manageable sample from the video matrix
max_frames = min(A.shape[1], 120)
A_sample = A[:, :max_frames]

L, S, history = solve_rpca(
    A_sample,
    max_iter=300,
    tol=1e-6,
    return_history=True,
    verbose=True,
)

recon_err = np.linalg.norm(A_sample - L - S, "fro") / np.linalg.norm(A_sample, "fro")
print(f"Converged in {len(history)} iterations")
print(f"Relative reconstruction error: {recon_err:.3e}")
print(f"Estimated rank(L): {np.linalg.matrix_rank(L)}")
print(f"S sparsity ratio: {np.mean(np.abs(S) > 0):.4f}")



iter=   1 err=3.848e-02 rank(L)=1 sparsity(S)=0.0000
iter=  10 err=3.243e-02 rank(L)=1 sparsity(S)=0.0500
iter=  20 err=1.554e-04 rank(L)=68 sparsity(S)=0.8924
iter=  30 err=6.866e-07 rank(L)=70 sparsity(S)=0.9194
Converged in 30 iterations
Relative reconstruction error: 6.866e-07
Estimated rank(L): 70
S sparsity ratio: 0.9194
